## Data Download

In [1]:
# 다운로드 받는 코드
import kagglehub
import os
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("andrezaza/clapper-massive-rotten-tomatoes-movies-and-reviews")
path = path.replace('\\', '/')

review_df = pd.read_csv(os.path.join(path, os.listdir(path)[0]))
movie_df = pd.read_csv(os.path.join(path, os.listdir(path)[1]))

print("Path to dataset files:", path)
print(movie_df.shape)
print(review_df.shape)

/home/shin/anaconda3/envs/rec-modeling-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/shin/.cache/kagglehub/datasets/andrezaza/clapper-massive-rotten-tomatoes-movies-and-reviews/versions/4
(143258, 16)
(1444963, 11)


## Movie feature

In [2]:
movie_df.head()

,id,title,audienceScore,tomatoMeter,rating,ratingContents,releaseDateTheaters,releaseDateStreaming,runtimeMinutes,genre,originalLanguage,director,writer,boxOffice,distributor,soundMix
0,space-zombie-bingo,Space Zombie Bingo!,50.0,NaN,NaN,NaN,NaN,2018-08-25,75.0,"Comedy, Horror, Sci-fi",English,George Ormrod,"George Ormrod,John Sabotta",NaN,NaN,NaN
1,the_green_grass,The Green Grass,NaN,NaN,NaN,NaN,NaN,2020-02-11,114.0,Drama,English,Tiffany Edwards,Tiffany Edwards,NaN,NaN,NaN
2,love_lies,"Love, Lies",43.0,NaN,NaN,NaN,NaN,NaN,120.0,Drama,Korean,"Park Heung-Sik,Heung-Sik Park","Ha Young-Joon,Jeon Yun-su,Song Hye-jin",NaN,NaN,NaN
3,the_sore_losers_1997,Sore Losers,60.0,NaN,NaN,NaN,NaN,2020-10-23,90.0,"Action, Mystery & thriller",English,John Michael McCarthy,John Michael McCarthy,NaN,NaN,NaN
4,dinosaur_island_2002,Dinosaur Island,70.0,NaN,NaN,NaN,NaN,2017-03-27,80.0,"Fantasy, Adventure, Animation",English,Will Meugniot,John Loy,NaN,NaN,NaN


In [3]:
movie_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 143258 entries, 0 to 143257
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    143258 non-null  object 
 1   title                 142891 non-null  object 
 2   audienceScore         73248 non-null   float64
 3   tomatoMeter           33877 non-null   float64
 4   rating                13991 non-null   object 
 5   ratingContents        13991 non-null   object 
 6   releaseDateTheaters   30773 non-null   object 
 7   releaseDateStreaming  79420 non-null   object 
 8   runtimeMinutes        129431 non-null  float64
 9   genre                 132175 non-null  object 
 10  originalLanguage      129400 non-null  object 
 11  director              139041 non-null  object 
 12  writer                90116 non-null   object 
 13  boxOffice             14743 non-null   object 
 14  distributor           23001 non-null   object 
 15  

### audienceScore
- 일반인의 평균 평가 점수
- 0-1로 정규화, 평균값으로 fillna

In [4]:
movie_df["audienceScore"].describe()

count    73248.000000
mean        55.674967
std         24.553648
min          0.000000
25%         37.000000
50%         57.000000
75%         76.000000
max        100.000000
Name: audienceScore, dtype: float64

### tomatoMeter
- 전문 평가자들의 긍정 리뷰 비율
- 0-1로 정규화, 평균값으로 fillna

In [5]:
movie_df["tomatoMeter"].describe()

count    33877.000000
mean        65.770346
std         28.023203
min          0.000000
25%         45.000000
50%         73.000000
75%         89.000000
max        100.000000
Name: tomatoMeter, dtype: float64

### rating
- 영화 연령 등급 -> 연령등급 순으로 인코딩
    - G: 전체관람가 -> 1
    - PG: 부모 지도 하 관람 권장 -> 2
    - PG-13: 13세 이상 부모 지도 권장 -> 3
    - R: 17세 미만 보호자 동반 필수 -> 4
    - NC-17: 17세 미만 관람 불가 -> 5
    
    - TVY7: 7세 이상 어린이 대상 -> 0
    - TVG: 전체 관람가 -> 1
    - TVPG: 부모 지도하 관람 권장 -> 2
    - TV14: 14세 이상 시청 권장 -> 3
    - TVMA: 성인관람가 -> 5
- 인코딩 한것을 0-1 정규화, 평균으로 fillna

In [6]:
movie_df["rating"].describe()

count     13991
unique       10
top           R
freq       7734
Name: rating, dtype: object

In [7]:
movie_df["rating"].unique()

array([nan, 'PG-13', 'TVPG', 'R', 'PG', 'TV14', 'NC-17', 'TVG', 'TVMA',
       'TVY7', 'G'], dtype=object)

In [8]:
movie_df["rating"].value_counts()

rating
R        7734
PG-13    3446
PG       1911
TVPG      424
TV14      397
TVMA       57
NC-17      19
TVG         1
TVY7        1
G           1
Name: count, dtype: int64

### ratingContents
- rating 평가 근거
- one-hot encoding 방법밖에 없는것 같은데 너무 커서 삭제 -> 추가하면 좋은 feature 이긴 함

In [9]:
movie_df["ratingContents"].value_counts()   

ratingContents
['Language']                                                                          365
['V']                                                                                 155
['Some Language']                                                                     143
['Some Violence']                                                                     126
['L']                                                                                 105
                                                                                     ... 
['Some Drug Content', 'Sexuality', 'Pervasive Language', 'Strong Violence']             1
['Intense Sequences of Violence', 'Intense Sequences of Action', 'Sexual Content']      1
['Some Disturbing Sequences', 'Language']                                               1
['Strong Sexual Images', 'Language']                                                    1
['Partying', 'Brief Language', 'Sexual Content', 'Smoking', 'Some Violent Images']   

In [10]:
import ast

rating_contents = []
for lst in movie_df["ratingContents"].apply(lambda x: [] if pd.isna(x) else ast.literal_eval(x)).to_list():
    rating_contents += lst

In [11]:
len(rating_contents)

36381

In [12]:
pd.DataFrame(rating_contents).value_counts()

0                          
Language                       5809
Violence                       1929
Sexual Content                 1219
Some Violence                   931
Nudity                          913
                               ... 
Violent Sequences                 1
Violent Sports Action             1
Violent War Images                1
A Brief Substance Reference       1
A Brief Suggestive Comment        1
Name: count, Length: 2092, dtype: int64

### genre
- 영화 장르
- one-hot encoding
- 비슷한 장르 하나로 합치기
    - Sports & fitness 와 Sport
    - Musical 와 Music
    - Biography 와 History 와 War
    - LGBTQ+ 와 Gay & lesbian
    - Animation 와 Anime
    - Stand-up 와 Comedy 와 Entertainment
    - Special Interest 와 News 와 Short 와 Variety 와 Health & wellness 와 Faith & spirituality 와 Reality 와 Other

In [13]:
movie_df["genre"].value_counts()

genre
Drama                                                            27860
Documentary                                                      15162
Comedy                                                           11514
Mystery & thriller                                                7015
Comedy, Drama                                                     5479
                                                                 ...  
Action, Comedy, Fantasy, Horror, Sci-fi                              1
Documentary, Lgbtq+, Drama                                           1
Kids & family, Holiday, Musical, Comedy, Adventure, Animation        1
Drama, Romance, Sports & fitness                                     1
Crime, Documentary, History, Sports & fitness                        1
Name: count, Length: 2912, dtype: int64

In [14]:
movie_df["genre"].apply(lambda x: [] if pd.isna(x) else x.split(", ")).explode().value_counts()

genre
Drama                   55595
Comedy                  30076
Documentary             18907
Mystery & thriller      18191
Action                  11187
Horror                  11030
Romance                 10856
Adventure                7770
Crime                    6363
Fantasy                  4588
Sci-fi                   4435
Kids & family            3436
Animation                3314
Biography                3193
Western                  2846
Musical                  2504
History                  2259
War                      1818
Holiday                  1815
Music                    1468
Lgbtq+                   1339
Anime                     726
Stand-up                  221
Foreign                    77
Sports & fitness           64
Sports                     40
Nature                     25
Gay & lesbian              22
Other                      21
Short                      19
Variety                    10
Special interest            8
Health & wellness           4
Fait

### releaseDateTheaters
- 극장 개봉 날짜
- 연도만 추출한 다음 0-1로 정규화, 평균으로 fillna

### releaseDateStreaming
- releaseDateTheaters 와 같이 연도만 추출
- releaseDateTheaters 와 합쳐서 -> releaseDate로 만듬
- 0-1로 정규화, 평균으로 fillna

### runtimeMinutes
- 0-1로 정규화, 평균으로 fillna

In [15]:
movie_df["originalLanguage"].value_counts()

originalLanguage
English            85034
Spanish             4786
Japanese            3482
Hindi               3309
French (Canada)     3282
                   ...  
Yoruba                 1
Hawaiian               1
Bhojpuri               1
smi                    1
Aramaic                1
Name: count, Length: 112, dtype: int64

### originalLanguage -> language_category로 대체
- 한국어 까지만 사용하고 나머지는 other로 묶기
- one-hot encoding

In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 언어별 분포 계산
language_counts = movie_df["originalLanguage"].value_counts()
total_movies = len(movie_df)

# 누적 비율 계산
cumulative_percentages = (language_counts.cumsum() / total_movies * 100)

# 90% 기준으로 언어 분류
languages_to_keep = cumulative_percentages[cumulative_percentages <= 90].index
other_languages = language_counts[~language_counts.index.isin(languages_to_keep)]

# 새로운 Series 생성 (상위 언어 + Other)
plot_data = language_counts[languages_to_keep].copy()
plot_data['Other'] = other_languages.sum()

# 상세 정보 출력
print("90% 커버리지에 포함된 언어들:")
for lang in languages_to_keep:
    print(f"{lang}: {language_counts[lang]} movies ({language_counts[lang]/total_movies*100:.2f}%)")

print("\nOther 카테고리:")
print(f"Other: {plot_data['Other']} movies ({plot_data['Other']/total_movies*100:.2f}%)")

# 데이터프레임 변환 함수 (선택적)
def categorize_language(language):
    return language if language in languages_to_keep else 'Other'

# 원본 데이터프레임에 적용 (선택적)
movie_df['language_category'] = movie_df['originalLanguage'].apply(categorize_language)

90% 커버리지에 포함된 언어들:
English: 85034 movies (59.36%)
Spanish: 4786 movies (3.34%)
Japanese: 3482 movies (2.43%)
Hindi: 3309 movies (2.31%)
French (Canada): 3282 movies (2.29%)
Chinese: 3166 movies (2.21%)
French (France): 2760 movies (1.93%)
English (United Kingdom): 2553 movies (1.78%)
Italian: 2303 movies (1.61%)
German: 2155 movies (1.50%)
Korean: 1226 movies (0.86%)
Arabic: 938 movies (0.65%)
Spanish (Spain): 936 movies (0.65%)
Tamil: 909 movies (0.63%)
Russian: 898 movies (0.63%)
Portuguese (Brazil): 867 movies (0.61%)
Telugu: 774 movies (0.54%)
Malayalam: 642 movies (0.45%)
Unknown language: 528 movies (0.37%)
Dutch: 482 movies (0.34%)
English (Australia): 478 movies (0.33%)
Polish: 450 movies (0.31%)
Swedish: 432 movies (0.30%)
Hebrew: 428 movies (0.30%)
Turkish: 399 movies (0.28%)
Kannada: 396 movies (0.28%)
Danish: 370 movies (0.26%)
Tagalog: 369 movies (0.26%)
Persian: 362 movies (0.25%)
Filipino: 353 movies (0.25%)
Bangla: 302 movies (0.21%)
Thai: 275 movies (0.19%)
Norwegian: 

### boxOffice
- 극장 상영 수익
- 파싱 후 인코딩
- 0-1 정규화, 평균으로 fillna

In [17]:
movie_df["boxOffice"].value_counts()

boxOffice
$1.1M      118
$1.2M      100
$1.0M       97
$1.3M       97
$1.6M       70
          ... 
$60.5K       1
$179.7K      1
$259.7M      1
$432.5K      1
$248.8M      1
Name: count, Length: 4863, dtype: int64

### distributor
- 분포도가 애매해서 삭제
- 추후에 포함하면 좋을 feature임

In [18]:
movie_df["distributor"]

0               NaN
1               NaN
2               NaN
3               NaN
4               NaN
            ...    
143253    ADV Films
143254          NaN
143255          NaN
143256          NaN
143257          NaN
Name: distributor, Length: 143258, dtype: object

In [19]:
distributor_count_df = movie_df["distributor"].apply(lambda x: [] if pd.isna(x) else x.split(", ")).explode().value_counts()

In [20]:
total_movies = len(movie_df)

cumulative_percentages = (distributor_count_df.cumsum() / total_movies * 100)

# 90% 기준으로 언어 분류
distributor_to_keep = cumulative_percentages[cumulative_percentages <= 90].index
other_distributor = distributor_count_df[~distributor_count_df.index.isin(distributor_to_keep)]

# 새로운 Series 생성 (상위 언어 + Other)
plot_data = distributor_count_df[distributor_to_keep].copy()
plot_data['Other'] = other_distributor.sum()

# 상세 정보 출력
print("90% 커버리지에 포함된 언어들:")
for distributor in distributor_to_keep:
    print(f"{distributor}: {distributor_count_df[distributor]} movies ({distributor_count_df[distributor]/total_movies*100:.2f}%)")

print("\nOther 카테고리:")
# print(f"Other: {plot_data['Other']} movies ({plot_data
# ['Other']/total_movies*100:.2f}%)")


90% 커버리지에 포함된 언어들:
Paramount Pictures: 1116 movies (0.78%)
20th Century Fox: 861 movies (0.60%)
Universal Pictures: 854 movies (0.60%)
Warner Bros. Pictures: 724 movies (0.51%)
Metro-Goldwyn-Mayer: 722 movies (0.50%)
Columbia Pictures: 645 movies (0.45%)
IFC Films: 562 movies (0.39%)
United Artists: 496 movies (0.35%)
Warner Bros.: 486 movies (0.34%)
Lionsgate Films: 454 movies (0.32%)
Sony Pictures Classics: 422 movies (0.29%)
Sony Pictures Entertainment: 387 movies (0.27%)
Gravitas Ventures: 385 movies (0.27%)
Miramax Films: 363 movies (0.25%)
Magnolia Pictures: 320 movies (0.22%)
Vertical Entertainment: 310 movies (0.22%)
Buena Vista Pictures: 243 movies (0.17%)
Strand Releasing: 225 movies (0.16%)
RKO Radio Pictures: 223 movies (0.16%)
New Line Cinema: 189 movies (0.13%)
Saban Films: 184 movies (0.13%)
Kino Lorber: 184 movies (0.13%)
Focus Features: 180 movies (0.13%)
MGM/UA Home Entertainment Inc.: 180 movies (0.13%)
Fox: 179 movies (0.12%)
Criterion Collection: 172 movies (0.12%)

### soundMix
- 아래 표와 같이 그룹화
- one-hot encoding, nan 이면 모두 0

| 그룹명               | 포함 항목                                                                                                   |
|--------------------|---------------------------------------------------------------------------------------------------------|
| **Surround Sound** | `Surround`, `Dolby Digital Surround EX`, `Dolby Digital`, `Dolby Atmos`, `Dolby EX`, `DTS`, `DTS-ES`, `Digital 5.1` |
| **Stereo Sound**   | `Stereo`, `Dolby Stereo`, `Magnetic Stereo 6 Track`, `Magnetic Stereo 4 Track`, `Ultra-Stereo`, `Perspecta Stereo` |
| **Mono Sound**     | `Mono`                                                                                                   |
| **Dolby SR**       | `Dolby SR`, `Dolby SRD`                                                                                  |
| **Dolby Systems**  | `Dolby`, `Dolby A`                                                                                       |
| **Legacy Systems** | `Vitaphone`, `Fantasound`, `LC-Concept Digital Sound`, `CDS`, `Datasat`, `DVS`                           |


In [21]:
movie_df["soundMix"].info()

<class 'pandas.core.series.Series'>
RangeIndex: 143258 entries, 0 to 143257
Series name: soundMix
Non-Null Count  Dtype 
--------------  ----- 
15917 non-null  object
dtypes: object(1)
memory usage: 1.1+ MB


In [22]:
movie_df["soundMix"].apply(lambda x: [] if pd.isna(x) else x.split(", ")).explode().value_counts()

soundMix
Surround                     7279
Dolby Digital                4905
Stereo                       3284
DTS                          2004
SDDS                         1774
Dolby SR                     1518
Mono                         1324
Dolby Stereo                 1151
Dolby A                       937
Dolby SRD                     792
Dolby                         763
Dolby Atmos                   612
Datasat                       328
Magnetic Stereo 6 Track        88
Ultra-Stereo                   46
Dolby EX                       30
DTS-ES                         28
Digital 5.1                    23
Dolby Digital Surround EX      19
CDS                             6
Perspecta Stereo                4
Vitaphone                       3
Magnetic Stereo 4 Track         2
Fantasound                      2
LC-Concept Digital Sound        2
DVS                             1
Name: count, dtype: int64

## feature engineering

In [23]:
movie_preprocessed_df = pd.DataFrame()

### id

In [24]:
movie_preprocessed_df["id"] = movie_df["id"]

### title은 무시

### audienceScore

In [25]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
audience_score_mean = movie_df["audienceScore"].mean()
movie_preprocessed_df["audienceScore"] = movie_df["audienceScore"].fillna(audience_score_mean)
movie_preprocessed_df["audienceScore"] = scaler.fit_transform(movie_preprocessed_df[["audienceScore"]])

### tomatoMeter

In [26]:
scaler = MinMaxScaler()
audience_score_mean = movie_df["tomatoMeter"].mean()
movie_preprocessed_df["tomatoMeter"] = movie_df["tomatoMeter"].fillna(audience_score_mean)
movie_preprocessed_df["tomatoMeter"] = scaler.fit_transform(movie_preprocessed_df[["tomatoMeter"]])

### rating

In [27]:
rating_map = {
    'G': 1,
    'PG': 2,
    'PG-13': 3,
    'R': 4,
    'NC-17': 5,
    'TVY7': 0,
    'TVG': 1,
    'TVPG': 2,
    'TV14': 3,
    'TVMA': 5
}

movie_preprocessed_df["rating"] = movie_df["rating"].map(rating_map)
movie_preprocessed_df["rating"] = movie_preprocessed_df["rating"].fillna(movie_preprocessed_df["rating"].mean())

scaler = MinMaxScaler()
movie_preprocessed_df["rating"] = scaler.fit_transform(movie_preprocessed_df[["rating"]])

### ratingContents 은 무시

### releaseDateTheaters, releaseDateStreaming

In [28]:
movie_df['releaseDateTheaters'] = pd.to_datetime(movie_df['releaseDateTheaters'], errors='coerce')
movie_df['releaseDateStreaming'] = pd.to_datetime(movie_df['releaseDateStreaming'], errors='coerce')

# releaseDate 컬럼 생성
movie_preprocessed_df["releaseYear"] = movie_df.apply(
    lambda row: (
        pd.NaT if pd.isna(row['releaseDateTheaters']) and pd.isna(row['releaseDateStreaming'])
        else row['releaseDateTheaters'] if pd.isna(row['releaseDateStreaming'])
        else row['releaseDateStreaming'] if pd.isna(row['releaseDateTheaters'])
        else min(row['releaseDateTheaters'], row['releaseDateStreaming'])
    ).year,
    axis=1
)

movie_preprocessed_df["releaseYear"] = movie_preprocessed_df["releaseYear"].fillna(movie_preprocessed_df["releaseYear"].mean())
scaler = MinMaxScaler()
movie_preprocessed_df["releaseYear"] = scaler.fit_transform(movie_preprocessed_df[["releaseYear"]])

### runtimeMinutes

In [29]:
scaler = MinMaxScaler()
movie_preprocessed_df["runtimeMinutes"] = movie_df["runtimeMinutes"].fillna(movie_df["runtimeMinutes"].mean())
movie_preprocessed_df["runtimeMinutes"] = scaler.fit_transform(movie_preprocessed_df[["runtimeMinutes"]])

### genre

In [30]:
genre_map = {
    "Sport": "Sports & fitness",
    "Musical": "Music",
    "Biography": "History",
    "War": "History",
    "Gay & lesbian": "LGBTQ+",
    "Anime": "Animation",
    "Comedy": "Entertainment",
    "Stand-up": "Entertainment",
    "Special Interest": "Other",
    "News": "Other",
    "Short": "Other",
    "Variety": "Other",
    "Health & wellness": "Other",
    "Faith & spirituality": "Other",
    "Reality": "Other"
}

unique_genre = movie_df["genre"].apply(lambda x: [] if pd.isna(x) else x.split(", ")).explode().dropna().unique()
unique_genre = set([genre if genre not in genre_map else genre_map[genre] for genre in unique_genre])
unique_genre_map = {genre: i for i, genre in enumerate(unique_genre)}

unique_genre_map


{'Holiday': 0,
 'Western': 1,
 'Action': 2,
 'Foreign': 3,
 'Music': 4,
 'Fantasy': 5,
 'Adventure': 6,
 'History': 7,
 'Romance': 8,
 'Sports & fitness': 9,
 'Crime': 10,
 'LGBTQ+': 11,
 'Animation': 12,
 'Documentary': 13,
 'Sports': 14,
 'Other': 15,
 'Mystery & thriller': 16,
 'Sci-fi': 17,
 'Nature': 18,
 'Drama': 19,
 'Kids & family': 20,
 'Entertainment': 21,
 'Lgbtq+': 22,
 'Horror': 23,
 'Special interest': 24}

In [31]:
movie_df

,id,title,audienceScore,tomatoMeter,rating,ratingContents,releaseDateTheaters,releaseDateStreaming,runtimeMinutes,genre,originalLanguage,director,writer,boxOffice,distributor,soundMix,language_category
0,space-zombie-bingo,Space Zombie Bingo!,50.0,NaN,NaN,NaN,NaT,2018-08-25,75.0,"Comedy, Horror, Sci-fi",English,George Ormrod,"George Ormrod,John Sabotta",NaN,NaN,NaN,English
1,the_green_grass,The Green Grass,NaN,NaN,NaN,NaN,NaT,2020-02-11,114.0,Drama,English,Tiffany Edwards,Tiffany Edwards,NaN,NaN,NaN,English
2,love_lies,"Love, Lies",43.0,NaN,NaN,NaN,NaT,NaT,120.0,Drama,Korean,"Park Heung-Sik,Heung-Sik Park","Ha Young-Joon,Jeon Yun-su,Song Hye-jin",NaN,NaN,NaN,Korean
3,the_sore_losers_1997,Sore Losers,60.0,NaN,NaN,NaN,NaT,2020-10-23,90.0,"Action, Mystery & thriller",English,John Michael McCarthy,John Michael McCarthy,NaN,NaN,NaN,English
4,dinosaur_island_2002,Dinosaur Island,70.0,NaN,NaN,NaN,NaT,2017-03-27,80.0,"Fantasy, Adventure, Animation",English,Will Meugniot,John Loy,NaN,NaN,NaN,English
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143253,nadia_the_secret_of_blue_water_the_motion_pict...,Nadia: The Secret of Blue Water: The Motion Pi...,14.0,NaN,NaN,NaN,2002-08-27,NaT,90.0,"Action, Adventure, Anime",Japanese,Sho Aono,Kaoru Umeno,NaN,ADV Films,NaN,Japanese
143254,everyone_i_knew_and_loved,Everyone I Knew and Loved,NaN,NaN,NaN,NaN,NaT,NaT,99.0,Drama,English,Andrew Behringer,Erika Heidewald,NaN,NaN,NaN,English
143255,the-human-body,The Human Body,71.0,89.0,NaN,NaN,NaT,NaT,43.0,Documentary,English,Peter Georgi,Richard Dale,NaN,NaN,NaN,English
143256,flying_fists,Flying Fists,NaN,NaN,NaN,NaN,NaT,2006-11-21,63.0,Drama,English,Robert F. Hill,"Robert F. Hill,Basil Dickey",NaN,NaN,NaN,English


In [32]:
def map_genres(x):
    if pd.isna(x):
        return []
    return [genre_map[g] if g in genre_map else g for g in x.split(", ")]

# 장르 매핑과 원-핫 인코딩을 한번에 처리
genres_encode = movie_df["genre"].apply(lambda x: pd.Series([1 if genre in map_genres(x) else 0 for genre in unique_genre], index=unique_genre))
movie_preprocessed_df = pd.concat([movie_preprocessed_df, genres_encode], axis=1)

### boxOffice

In [33]:
def convert_boxoffice(value):
    if pd.isna(value):
        return None
    
    # 문자열에서 금액 부분만 추출
    value = str(value).replace('$', '').strip()
    
    # K(천), M(백만), B(십억) 변환
    multiplier = 1
    if value.endswith('K'):
        multiplier = 1000
        value = value[:-1]
    elif value.endswith('M'):
        multiplier = 1000000
        value = value[:-1]
    elif value.endswith('B'):
        multiplier = 1000000000
        value = value[:-1]
    
    try:
        # 콤마 제거하고 float로 변환 후 multiplier 적용
        return float(value.replace(',', '')) * multiplier
    except:
        return None

# 컬럼에 적용
movie_preprocessed_df['boxOffice'] = movie_df['boxOffice'].apply(convert_boxoffice)
scaler = MinMaxScaler()
movie_preprocessed_df["boxOffice"] = movie_preprocessed_df["boxOffice"].fillna(movie_preprocessed_df["boxOffice"].mean())
movie_preprocessed_df["boxOffice"] = scaler.fit_transform(movie_preprocessed_df[["boxOffice"]])

### distributor 는 삭제

### soundMix

In [34]:
sound_groups = {
   'Surround Sound': ['Surround', 'Dolby Digital Surround EX', 'Dolby Digital', 'Dolby Atmos', 
                     'Dolby EX', 'DTS', 'DTS-ES', 'Digital 5.1'],
   'Stereo Sound': ['Stereo', 'Dolby Stereo', 'Magnetic Stereo 6 Track', 'Magnetic Stereo 4 Track',
                    'Ultra-Stereo', 'Perspecta Stereo'],
   'Mono Sound': ['Mono'],
   'Dolby SR': ['Dolby SR', 'Dolby SRD'],
   'Dolby Systems': ['Dolby', 'Dolby A'],
   'Legacy Systems': ['Vitaphone', 'Fantasound', 'LC-Concept Digital Sound', 'CDS', 'Datasat', 'DVS']
}


# soundMix 값을 파싱하고 해당하는 그룹에 1 할당
def assign_sound_groups(sound_mix):
   if pd.isna(sound_mix):
       return pd.Series({group: 0 for group in sound_groups.keys()})
   sound_systems = sound_mix.split(", ")
   result = {group: 0 for group in sound_groups.keys()}
   
   for system in sound_systems:
       for group, systems in sound_groups.items():
           if any(s == system for s in systems):
               result[group] = 1
               break
   
   return pd.Series(result)

# 데이터프레임에 적용
sound_encoded = movie_df['soundMix'].apply(assign_sound_groups)
movie_preprocessed_df = pd.concat([movie_preprocessed_df, sound_encoded], axis=1)

In [35]:
movie_preprocessed_df.describe()

,audienceScore,tomatoMeter,rating,releaseYear,runtimeMinutes,Holiday,Western,Action,Foreign,Music,...,Lgbtq+,Horror,Special interest,boxOffice,Surround Sound,Stereo Sound,Mono Sound,Dolby SR,Dolby Systems,Legacy Systems
count,143258.000000,143258.000000,143258.000000,143258.000000,143258.000000,143258.000000,143258.000000,143258.000000,143258.000000,143258.000000,...,143258.000000,143258.000000,143258.000000,143258.000000,143258.000000,143258.000000,143258.000000,143258.000000,143258.000000,143258.000000
mean,0.556750,0.657703,0.679251,0.812883,0.034349,0.012669,0.019866,0.078090,0.000537,0.027656,...,0.009347,0.076994,0.000056,0.021313,0.081224,0.031133,0.009242,0.015950,0.011860,0.002387
std,0.175571,0.136272,0.047830,0.116965,0.009906,0.111844,0.139541,0.268314,0.023178,0.163987,...,0.096226,0.266583,0.007473,0.017991,0.273180,0.173677,0.095691,0.125283,0.108255,0.048802
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.556750,0.657703,0.679251,0.812883,0.031123,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.021313,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.556750,0.657703,0.679251,0.812883,0.034349,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.021313,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.580000,0.657703,0.679251,0.884615,0.037051,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.021313,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [36]:
movie_preprocessed_df.to_csv("movie_preprocessed.csv", index=False)